In this notebook, we compute reference scores used to interpret Granuscore outputs via percentiles. We use the sentence-transformers/wikipedia-en-sentences dataset as the reference corpus.

In [1]:
from datasets import load_dataset

ds = load_dataset("sentence-transformers/wikipedia-en-sentences", split="train")
ds = ds.shuffle(seed=0)

In [2]:
def extract_wikipedia_sentences(
    dataset,
    n_sentences=100_000,
    min_len=5,
    max_len=40,
):
    sentences = []
    for entry in dataset:
        sent = entry["sentence"]
        if len(sentences) >= n_sentences:
            break
        n_words = len(sent.split())
        if min_len <= n_words <= max_len:
            sentences.append(sent)

    return sentences

texts = extract_wikipedia_sentences(
    ds,
    n_sentences=100_000,
    min_len=5,
    max_len=40,
)

In [3]:
from granuscore import GranuScore
import numpy as np


granu_score = GranuScore()
scores = granu_score(texts, encoding_batch_size=128, batch_size=128, pooling='mean', show_progress_bar=True)
np.save("scores-wiki_sentences-100k-mean.npy", scores)

[GranuScorer] macOS detected → using FAISS single-thread mode.


Granuscore: 100%|██████████| 782/782 [36:41<00:00,  2.82s/it]


In [4]:
from scipy.stats import percentileofscore
import random


def sample_words_near_percentile(
    texts,
    scores,
    reference_scores,
    target_p: int,
    band: float = 2.5,   # ±2.5 percentile points
    k: int = 10,
    seed: int = 0,
):
    rng = random.Random(seed)

    percentiles = [
        percentileofscore(reference_scores, sc)
        for sc in scores
    ]

    candidates = [
        texts[i]
        for i, p in enumerate(percentiles)
        if abs(p - target_p) <= band
    ]

    if len(candidates) <= k:
        return candidates

    return rng.sample(candidates, k)

In [8]:
def pretty_print_reference_words(reference_words):
    for p in sorted(reference_words):
        print(f"\nPercentile {p}")
        print("-" * (len(str(p)) + 11))
        for i, word in enumerate(reference_words[p], 1):
            print(f"  {i:>2}. {word}")

reference_words = {
    p: sample_words_near_percentile(
        texts=texts,
        scores=scores,
        reference_scores=scores,
        target_p=p,
        band=3,
        k=10,
        seed=0,
    )
    for p in [16.67, 50, 83.33]
}

pretty_print_reference_words(reference_words)


Percentile 16.67
----------------
   1. For certain other particular topics, see the articles listed in the adjacent box.
   2. 2017 Jerusalem Light Rail stabbing was a stabbing attack and suspected act of terrorism that occurred on Good Friday, April 14, 2017, on Jerusalem Light Rail's car.
   3. Prevailing winds can have differences due to the uneven heating of the Earth.
   4. Josh Janniere (born November 4, 1992) is a Canadian soccer player who most recently played for Colorado Rapids in Major League Soccer.
   5. An older binomial name for this species is Eupatorium rugosum, but the genus Eupatorium has undergone taxonomic revision by botanists, and a number of the species that were once included in it have been moved to other genera.
   6. Although founded in the United States, with offices in New York City, the MLA's membership, concerns, reputation, and influence are international in scope.
   7. Flotsam and Jetsam have released twelve studio albums in their career (the latest